# 预期ST因子测试

因子定义：连续两年净利润为负（老规则）

策略逻辑：
1. 每天从全市场筛选总市值最小的500只股票（尾部500）
2. 在尾部500中，筛选连续两年净利润为负的股票
3. 等权持有所有满足条件的股票
4. 每天调仓

In [ ]:
from bigmodule import M, I
import dai
import pandas as pd


def m5_initialize_bigquant_run(context):
    from bigtrader.finance.commission import PerOrder
    context.set_commission(PerOrder(buy_cost=0.0003, sell_cost=0.0013, min_cost=5))


def m5_before_trading_start_bigquant_run(context, data):
    pass


def m5_handle_tick_bigquant_run(context, tick):
    pass


def m5_handle_data_bigquant_run(context, data):
    today_df = context.data[context.data["date"] == data.current_dt.strftime("%Y-%m-%d")]
    target_instruments = set(today_df["instrument"])
    holding_instruments = set(context.get_account_positions().keys())

    for instrument in holding_instruments - target_instruments:
        context.order_target_percent(instrument, 0)

    for i, x in today_df.iterrows():
        position = 0.0 if pd.isnull(x.position) else float(x.position)
        context.order_target_percent(x.instrument, position)


def m5_handle_trade_bigquant_run(context, trade):
    pass


def m5_handle_order_bigquant_run(context, order):
    pass


def m5_after_trading_bigquant_run(context, data):
    pass


# ========== 数据准备 ==========
# 预期ST因子：连续两年净利润为负（老规则）
# 全部逻辑下推到 SQL 层，避免 Python 层过滤和计算

stock_sql = """
WITH base AS (
    SELECT
        date,
        instrument,
        total_market_cap,
        net_profit_ttm,
        m_lag(net_profit_ttm, 250) AS net_profit_ttm_ly
    FROM cn_stock_prefactors_community
    JOIN cn_stock_factors_financial_extend USING (date, instrument)
    WHERE
        st_status = 0
        AND suspended = 0
        AND is_bz50 = 0
        AND list_days > 365
    QUALIFY
        ROW_NUMBER() OVER (PARTITION BY date ORDER BY total_market_cap ASC) <= 500
),
filtered AS (
    SELECT * FROM base
    WHERE net_profit_ttm < 0 AND net_profit_ttm_ly < 0
)
SELECT
    date,
    instrument,
    total_market_cap,
    -total_market_cap AS score,
    1.0 / c_sum(1) AS position
FROM filtered
ORDER BY date, instrument
"""

print("正在查询数据...")
filtered_df = dai.query(stock_sql, filters={"date": ["2020-01-01", "2026-12-31"]}).df()
print(f"满足预期ST条件的记录数：{len(filtered_df)}")
print(f"每日平均持仓数量：{filtered_df.groupby('date')['instrument'].count().mean():.1f}")

stock_data_ds = dai.DataSource.write_bdb(filtered_df)

# ========== 回测 ==========
start_date = '2021-01-01'
end_date = '2026-04-07'

m5 = M.bigtrader.v30(
    data=stock_data_ds,
    start_date=start_date,
    end_date=end_date,
    initialize=m5_initialize_bigquant_run,
    before_trading_start=m5_before_trading_start_bigquant_run,
    handle_tick=m5_handle_tick_bigquant_run,
    handle_data=m5_handle_data_bigquant_run,
    handle_trade=m5_handle_trade_bigquant_run,
    handle_order=m5_handle_order_bigquant_run,
    after_trading=m5_after_trading_bigquant_run,
    capital_base=1000000,
    frequency="daily",
    product_type="股票",
    rebalance_period_type="交易日",
    rebalance_period_days="1",
    rebalance_period_roll_forward=True,
    backtest_engine_mode="标准模式",
    before_start_days=0,
    volume_limit=1,
    order_price_field_buy="open",
    order_price_field_sell="open",
    benchmark="沪深300指数",
    plot_charts=True,
    debug=False,
    backtest_only=False,
    m_name="m5"
)

In [ ]:
# ========== 导出交易记录CSV ==========
trades_df = m5.raw_perf.read()['transactions']

output_records = []
holdings = {}

for idx, row in trades_df.iterrows():
    instrument = row['symbol']
    dt = pd.to_datetime(row['dt']).strftime('%Y-%m-%d')
    amount = row['amount']
    price = row['price']
    
    if amount > 0:
        holdings[instrument] = {'buy_date': dt, 'buy_price': price}
    elif amount < 0 and instrument in holdings:
        buy_info = holdings.pop(instrument)
        pnl = (price - buy_info['buy_price']) / buy_info['buy_price']
        output_records.append({
            '股票代码': instrument.split('.')[0],
            '买入日期': buy_info['buy_date'],
            '卖出日期': dt,
            '买入价格(前复权)': round(buy_info['buy_price'], 2),
            '卖出价格(前复权)': round(price, 2),
            '涨幅': round(pnl, 4)
        })

output_df = pd.DataFrame(output_records)
output_df = output_df.sort_values('卖出日期', ascending=False)

output_path = './strategy/预期ST_bigquant交易记录.csv'
output_df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"交易记录已保存到：{output_path}")
print(f"共 {len(output_df)} 条交易记录")
output_df.head(20)